# Setup

In [1]:
import pandas as pd
import numpy as np
import random
import warnings
import torch
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, brier_score_loss, confusion_matrix,
)
from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    equalized_odds_difference,
)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

USE_GPU = torch.cuda.is_available()
print(f"CUDA available: {USE_GPU}")
if USE_GPU:
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
Using GPU: NVIDIA GeForce RTX 4060 Laptop GPU


### Reverse one-hot encoding

In [2]:
def reverse_onehot(df: pd.DataFrame, group_map: dict[str, str]) -> pd.DataFrame:
    df = df.copy()
    for prefix, suffixes in group_map.items():
        ohe_cols = [f"{prefix}={s}" for s in suffixes]
        present = [c for c in ohe_cols if c in df.columns]
        if not present:
            continue
        cat_values = df[present].idxmax(axis=1).str.replace(f"{prefix}=", "", regex=False)
        all_zero_mask = df[present].sum(axis=1) == 0
        cat_values[all_zero_mask] = "Unknown"
        df[prefix] = cat_values.astype(str)
        df = df.drop(columns=present)
    return df


def detect_onehot_groups(columns: list[str]) -> dict[str, list[str]]:
    groups = {}
    for col in columns:
        if "=" in col:
            prefix, suffix = col.rsplit("=", 1)
            groups.setdefault(prefix, []).append(suffix)
    return {k: v for k, v in groups.items() if len(v) >= 2}

### Model building + utility metrics

In [3]:
# sklearn pipeline + random forest classifier
def make_model(X: pd.DataFrame) -> Pipeline:
    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = X.select_dtypes(include=["number"]).columns.tolist()
    preprocess = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
        ("num", "passthrough", num_cols),
    ])
    return Pipeline([
        ("preprocess", preprocess),
        ("model", RandomForestClassifier(random_state=SEED, n_jobs=-1)),
    ])


def compute_utility(model, X_test, y_test) -> dict:
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    return {
        "AUROC": roc_auc_score(y_test, y_proba), # higher is better
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "Balanced_Acc": balanced_accuracy_score(y_test, y_pred),
        "Brier": brier_score_loss(y_test, y_proba), # lower is better
    }

### Fairness metrics (replaces AIF360 with MetricFrame)

In [4]:
def compute_fairness(y_test, y_pred, y_proba, sensitive_features,
                     pos_label=1) -> dict:
    def fpr(y_true, y_pred):
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        return fp / (fp + tn) if (fp + tn) > 0 else 0.0

    def tpr(y_true, y_pred):
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        return tp / (tp + fn) if (tp + fn) > 0 else 0.0

    def tnr(y_true, y_pred):
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        return tn / (tn + fp) if (tn + fp) > 0 else 0.0

    metric_fns = {
        "TPR (Sensitivity)": tpr,
        "TNR (Specificity)": tnr,
        "FPR": fpr,
        "Precision": lambda y, yp: precision_score(y, yp, zero_division=0),
        "Recall": lambda y, yp: recall_score(y, yp, zero_division=0),
        "F1": lambda y, yp: f1_score(y, yp, zero_division=0),
        "Positive_Rate": lambda y, yp: np.mean(yp),
    }

    mf = MetricFrame(
        metrics=metric_fns,
        y_true=y_test,
        y_pred=y_pred,
        sensitive_features=sensitive_features,
    )

    dp_diff = demographic_parity_difference(
        y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_features
    )

    eo_diff = equalized_odds_difference(
        y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_features
    )

    return {
        "DP_difference": dp_diff,
        "EO_difference": eo_diff,
        "per_group": mf.by_group,
        "overall": mf.overall,
        "difference": mf.difference(),
        "ratio": mf.ratio(),
    }


def ftu_flip_rate(model, X, protected_col, values=("1.0", "0.0")) -> float:
    X0 = X.copy(); X0[protected_col] = values[0]
    X1 = X.copy(); X1[protected_col] = values[1]
    return float(np.mean(model.predict(X0) != model.predict(X1)))

### Intersectional fairness

In [5]:
# FPR, TPR, F1 for Race x Sex
def compute_intersectional(y_test, y_pred, race_col, sex_col) -> pd.DataFrame:
    combined = race_col.astype(str) + "_" + sex_col.astype(str)

    def fpr(y_true, y_pred):
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp = cm[0, 0], cm[0, 1]
        return fp / (fp + tn) if (fp + tn) > 0 else 0.0

    def tpr(y_true, y_pred):
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        fn, tp = cm[1, 0], cm[1, 1]
        return tp / (tp + fn) if (tp + fn) > 0 else 0.0

    mf = MetricFrame(
        metrics={
            "FPR": fpr,
            "TPR": tpr,
            "F1": lambda y, yp: f1_score(y, yp, zero_division=0),
            "Count": lambda y, yp: len(y),
        },
        y_true=y_test,
        y_pred=y_pred,
        sensitive_features=combined,
    )
    return mf.by_group

### SDMetrics

In [6]:
def compute_sdmetrics(real_df, synth_df, metadata, target_col,
                      real_train_df=None, dcr_subsample=2000):
    from sdmetrics.reports.single_table import QualityReport, DiagnosticReport
    from sdmetrics.single_table import (
        KSComplement, TVComplement, CSTest,
        CorrelationSimilarity, ContinuousKLDivergence,
        ContingencySimilarity, DiscreteKLDivergence,
        CategoryCoverage, RangeCoverage, BoundaryAdherence,
        StatisticSimilarity, MissingValueSimilarity, TableStructure,
        GMLogLikelihood,
        NewRowSynthesis,
        DCRBaselineProtection, DCROverfittingProtection,
        BinaryLogisticRegression, BinaryDecisionTreeClassifier,
        BinaryAdaBoostClassifier, BinaryMLPClassifier,
    )
    from sdmetrics.single_table.privacy import DisclosureProtectionEstimate
    from sdmetrics.single_table import (
        CategoricalCAP, CategoricalZeroCAP, CategoricalGeneralizedCAP,
    )
    from sdmetrics.single_table import LogisticDetection, SVCDetection

    results = {}

    # split columns by type
    num_cols = [c for c, m in metadata["columns"].items()
                if m.get("sdtype") == "numerical"]
    cat_cols = [c for c, m in metadata["columns"].items()
                if m.get("sdtype") in ("categorical", "boolean")]

    # convert 'object' to 'category', unsigned ints to int64
    def align_df(df):
        df = df.copy()
        for col in df.columns:
            if df[col].dtype == "object":
                df[col] = df[col].astype("category")
            elif df[col].dtype.kind == "u":
                df[col] = df[col].astype("int64")
        return df

    # align real and synthetic so categories match across both
    def align_pair(r, s):
        r, s = align_df(r), align_df(s)
        for col in r.columns:
            if hasattr(r[col], "cat"):
                union = sorted(
                    set(r[col].cat.categories) | set(s[col].cat.categories),
                    key=str,
                )
                r[col] = r[col].cat.set_categories(union)
                s[col] = s[col].cat.set_categories(union)
        return r, s

    real_a, synth_a = align_pair(real_df, synth_df)

    # subset metadata
    def sub_meta(cols):
        return {"columns": {c: metadata["columns"][c]
                            for c in cols if c in metadata["columns"]}}

    # Reports
    q = QualityReport()
    q.generate(real_df, synth_df, metadata, verbose=False)
    results["QualityReport_overall"] = q.get_score()
    props = q.get_properties().set_index("Property")
    results["QualityReport_column_shapes"] = props.loc["Column Shapes", "Score"]
    results["QualityReport_column_pair_trends"] = props.loc["Column Pair Trends", "Score"]

    d = DiagnosticReport()
    d.generate(real_df, synth_df, metadata, verbose=False)
    results["DiagnosticReport_overall"] = d.get_score()

    # individual column metrics
    if num_cols:
        try:
            results["KSComplement"] = KSComplement.compute(
                real_data=real_df, synthetic_data=synth_df, metadata=metadata)
        except Exception:
            pass
    if cat_cols:
        try:
            results["TVComplement"] = TVComplement.compute(
                real_data=real_df, synthetic_data=synth_df, metadata=metadata)
        except Exception:
            pass
        try:
            results["CSTest"] = CSTest.compute(
                real_data=real_df, synthetic_data=synth_df, metadata=metadata)
        except Exception:
            pass

    # joint distribution
    if len(num_cols) >= 2:
        nm = sub_meta(num_cols)
        try:
            results["CorrelationSimilarity"] = CorrelationSimilarity.compute(
                real_data=real_df[num_cols], synthetic_data=synth_df[num_cols],
                metadata=nm)
        except Exception:
            pass
        try:
            results["ContinuousKLDivergence"] = ContinuousKLDivergence.compute(
                real_data=real_df[num_cols], synthetic_data=synth_df[num_cols],
                metadata=nm)
        except Exception:
            pass

    if len(cat_cols) >= 2:
        cm2 = sub_meta(cat_cols[:2])
        try:
            results["ContingencySimilarity"] = ContingencySimilarity.compute(
                real_data=real_df[cat_cols[:2]],
                synthetic_data=synth_df[cat_cols[:2]], metadata=cm2)
        except Exception:
            pass
        try:
            results["DiscreteKLDivergence"] = DiscreteKLDivergence.compute(
                real_data=real_df[cat_cols[:2]],
                synthetic_data=synth_df[cat_cols[:2]], metadata=cm2)
        except Exception:
            pass

    # coverage and structure
    if cat_cols:
        try:
            results["CategoryCoverage"] = CategoryCoverage.compute(
                real_data=real_df[cat_cols], synthetic_data=synth_df[cat_cols],
                metadata=sub_meta(cat_cols))
        except Exception:
            pass
    if num_cols:
        nm = sub_meta(num_cols)
        try:
            results["RangeCoverage"] = RangeCoverage.compute(
                real_data=real_df[num_cols], synthetic_data=synth_df[num_cols],
                metadata=nm)
        except Exception:
            pass
        try:
            results["BoundaryAdherence"] = BoundaryAdherence.compute(
                real_data=real_df[num_cols], synthetic_data=synth_df[num_cols],
                metadata=nm)
        except Exception:
            pass
        try:
            results["StatisticSimilarity"] = StatisticSimilarity.compute(
                real_data=real_df[num_cols], synthetic_data=synth_df[num_cols],
                metadata=nm)
        except Exception:
            pass
        try:
            results["MissingValueSimilarity"] = MissingValueSimilarity.compute(
                real_data=real_df[num_cols], synthetic_data=synth_df[num_cols],
                metadata=nm)
        except Exception:
            pass

    try:
        results["TableStructure"] = TableStructure.compute(
            real_data=real_df, synthetic_data=synth_df, metadata=metadata)
    except Exception:
        pass

    if num_cols:
        try:
            results["GMLogLikelihood"] = GMLogLikelihood.compute(
                real_data=real_df[num_cols], synthetic_data=synth_df[num_cols],
                metadata=sub_meta(num_cols))
        except Exception:
            pass

    # detection (can a classifier tell real from synthetic)
    det_n = min(5000, len(real_df), len(synth_df))
    r_det = real_a.sample(n=det_n, random_state=42) if len(real_a) > det_n else real_a
    s_det = synth_a.sample(n=det_n, random_state=42) if len(synth_a) > det_n else synth_a

    try:
        results["LRDetection"] = LogisticDetection.compute(
            real_data=r_det, synthetic_data=s_det, metadata=metadata)
    except Exception:
        pass
    try:
        results["SVCDetection"] = SVCDetection.compute(
            real_data=r_det, synthetic_data=s_det, metadata=metadata)
    except Exception:
        pass

    # ML efficacy (train-on-synthetic, test-on-real)
    try:
        # build from raw DataFrames, skip align_pair because it causes Categorical dtype issues with TVAE-generated categories
        real_eff = real_df.copy()
        synth_eff = synth_df.copy()

        # convert target to int (SDMetrics uses pos_label as int)
        for d in (real_eff, synth_eff):
            d[target_col] = d[target_col].astype(str).map(
                {"0.0": 0, "1.0": 1, "0": 0, "1": 1}).astype(int)

        # categorical cols as plain strings instead of pd.Categorical
        for col in real_eff.columns:
            if col != target_col and real_eff[col].dtype.name in ("object", "category"):
                real_eff[col] = real_eff[col].astype(str)
                synth_eff[col] = synth_eff[col].astype(str)

        # filter to rows where all category values exist in both datasets
        for col in real_eff.columns:
            if col != target_col and real_eff[col].dtype == "object":
                common = set(real_eff[col].unique()) & set(synth_eff[col].unique())
                real_eff = real_eff[real_eff[col].isin(common)]
                synth_eff = synth_eff[synth_eff[col].isin(common)]

        eff_meta = {"columns": {c: dict(v) for c, v in metadata["columns"].items()}}
        eff_meta["columns"][target_col] = {"sdtype": "numerical"}

        results["LRClassifierEfficacy"] = BinaryLogisticRegression.compute(
            test_data=real_eff, train_data=synth_eff,
            target=target_col, metadata=eff_meta)
        results["DTClassifierEfficacy"] = BinaryDecisionTreeClassifier.compute(
            test_data=real_eff, train_data=synth_eff,
            target=target_col, metadata=eff_meta)
        results["AdaBoostEfficacy"] = BinaryAdaBoostClassifier.compute(
            test_data=real_eff, train_data=synth_eff,
            target=target_col, metadata=eff_meta)
        results["MLPEfficacy"] = BinaryMLPClassifier.compute(
            test_data=real_eff, train_data=synth_eff,
            target=target_col, metadata=eff_meta)
    except Exception as e:
        results["ML_Efficacy_Error"] = str(e)
        
    # privacy
    try:
        results["NewRowSynthesis"] = NewRowSynthesis.compute(
            real_data=real_df, synthetic_data=synth_df, metadata=metadata,
            synthetic_sample_size=min(1000, len(synth_df)))
    except Exception as e:
        results["NewRowSynthesis"] = f"Error: {e}"

    key_fields = [c for c in cat_cols if c != target_col][:3] + num_cols[:2]
    sensitive = [target_col]
    try:
        results["DisclosureProtectionEstimate"] = DisclosureProtectionEstimate.compute(
            real_data=real_df, synthetic_data=synth_df,
            known_column_names=key_fields, sensitive_column_names=sensitive,
            continuous_column_names=num_cols,
            num_rows_subsample=min(1000, len(real_df)), num_iterations=3)
    except Exception as e:
        results["DisclosureProtectionEstimate"] = f"Error: {e}"

    try:
        results["DCRBaselineProtection"] = DCRBaselineProtection.compute(
            real_data=real_df, synthetic_data=synth_df,
            metadata=metadata, num_rows_subsample=dcr_subsample)
    except Exception as e:
        results["DCRBaselineProtection"] = f"Error: {e}"

    if real_train_df is not None:
        train_idx = real_df.index.intersection(real_train_df.index)
        r_train = real_df.loc[train_idx]
        r_valid = real_df.loc[real_df.index.difference(train_idx)]
        dcr_n = min(dcr_subsample, len(r_train), len(r_valid), len(synth_df))
        try:
            results["DCROverfittingProtection"] = DCROverfittingProtection.compute(
                real_training_data=r_train, synthetic_data=synth_df,
                real_validation_data=r_valid, metadata=metadata,
                num_rows_subsample=dcr_n)
        except Exception as e:
            results["DCROverfittingProtection"] = f"Error: {e}"

    cat_key = [c for c in key_fields if c in cat_cols]
    cat_sens = [c for c in sensitive if c in cat_cols]
    if cat_key and cat_sens:
        try:
            results["CategoricalCAP"] = CategoricalCAP.compute(
                real_data=real_df, synthetic_data=synth_df,
                key_fields=cat_key, sensitive_fields=cat_sens)
        except Exception:
            pass
        try:
            results["CategoricalZeroCAP"] = CategoricalZeroCAP.compute(
                real_data=real_df, synthetic_data=synth_df,
                key_fields=cat_key, sensitive_fields=cat_sens)
        except Exception:
            pass
        try:
            results["CategoricalGeneralizedCAP"] = CategoricalGeneralizedCAP.compute(
                real_data=real_df, synthetic_data=synth_df,
                key_fields=cat_key, sensitive_fields=cat_sens)
        except Exception:
            pass

    return results

### Load MEPS

In [7]:
from aif360.datasets import MEPSDataset19

meps_aif = MEPSDataset19()
df_raw = meps_aif.convert_to_dataframe()[0]

print(f"Raw shape: {df_raw.shape}")
print(f"{df_raw['RACE'].value_counts().to_string()}")
print(f"{df_raw['UTILIZATION'].value_counts().to_string()}")

# reverse one-hot encoding
onehot_groups = detect_onehot_groups(df_raw.columns.tolist())
print(f"\nDetected {len(onehot_groups)} one-hot groups: {list(onehot_groups.keys())}")

df = reverse_onehot(df_raw, onehot_groups)

# convert RACE and UTILIZATION to string
df["RACE"] = df["RACE"].astype(str)
df["UTILIZATION"] = df["UTILIZATION"].astype(str)

# convert float columns but actually categorical (0/1) to string
KEEP_NUMERIC = {"AGE", "PCS42", "MCS42", "K6SUM42"}
for col in df.columns:
    if col in KEEP_NUMERIC:
        continue
    if df[col].dtype in [np.float64, np.float32] and df[col].nunique() <= 10:
        df[col] = df[col].astype(int).astype(str)

print(f"After reverse encoding: {df.shape}")
print(f"Numeric columns: {df.select_dtypes(include='number').columns.tolist()}")
print(f"Categorical columns: {df.select_dtypes(include='object').columns.tolist()}")
print(f"\nSample:\n{df.head()}")

Raw shape: (15830, 139)
RACE
0.0    10174
1.0     5656
UTILIZATION
0.0    13112
1.0     2718

Detected 36 one-hot groups: ['REGION', 'SEX', 'MARRY', 'FTSTU', 'ACTDTY', 'HONRDC', 'RTHLTH', 'MNHLTH', 'HIBPDX', 'CHDDX', 'ANGIDX', 'MIDX', 'OHRTDX', 'STRKDX', 'EMPHDX', 'CHBRON', 'CHOLDX', 'CANCERDX', 'DIABDX', 'JTPAIN', 'ARTHDX', 'ARTHTYPE', 'ASTHDX', 'ADHDADDX', 'PREGNT', 'WLKLIM', 'ACTLIM', 'SOCLIM', 'COGLIM', 'DFHEAR42', 'DFSEE42', 'ADSMOK42', 'PHQ242', 'EMPST', 'POVCAT', 'INSCOV']
After reverse encoding: (15830, 42)
Numeric columns: ['AGE', 'PCS42', 'MCS42', 'K6SUM42']
Categorical columns: ['RACE', 'UTILIZATION', 'REGION', 'SEX', 'MARRY', 'FTSTU', 'ACTDTY', 'HONRDC', 'RTHLTH', 'MNHLTH', 'HIBPDX', 'CHDDX', 'ANGIDX', 'MIDX', 'OHRTDX', 'STRKDX', 'EMPHDX', 'CHBRON', 'CHOLDX', 'CANCERDX', 'DIABDX', 'JTPAIN', 'ARTHDX', 'ARTHTYPE', 'ASTHDX', 'ADHDADDX', 'PREGNT', 'WLKLIM', 'ACTLIM', 'SOCLIM', 'COGLIM', 'DFHEAR42', 'DFSEE42', 'ADSMOK42', 'PHQ242', 'EMPST', 'POVCAT', 'INSCOV']

Sample:
    AGE R

### Train/test split + real data baseline

In [8]:
TARGET = "UTILIZATION"
PROTECTED = "RACE"

X = df.drop(columns=[TARGET])
y = (df[TARGET] == "1.0").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y,
)

# train real-data baseline model
model_real = make_model(X_train)
model_real.fit(X_train, y_train)
y_pred_real = model_real.predict(X_test)
y_proba_real = model_real.predict_proba(X_test)[:, 1]

print("Real data baseline")
utility_real = compute_utility(model_real, X_test, y_test)
for k, v in utility_real.items():
    print(f"{k}: {v:.4f}")

fairness_real = compute_fairness(
    y_test, y_pred_real, y_proba_real,
    sensitive_features=X_test[PROTECTED],
)
print(f"\nDP difference: {fairness_real['DP_difference']:.4f}")
print(f"EO difference: {fairness_real['EO_difference']:.4f}")
print(f"\nPer-group metrics (Real):")
print(fairness_real["per_group"].to_string())

ftu_real = ftu_flip_rate(model_real, X_test, PROTECTED, values=("1.0", "0.0"))
print(f"\nFTU flip rate: {ftu_real:.4f}")

# intersectional (RACE x SEX)
SEX_COL = "SEX" if "SEX" in X_test.columns else None
if SEX_COL:
    print(f"\nIntersectional (RACE x SEX):")
    inter_real = compute_intersectional(
        y_test, y_pred_real, X_test[PROTECTED], X_test[SEX_COL]
    )
    print(inter_real.to_string())

Real data baseline
AUROC: 0.8426
Precision: 0.6264
Recall: 0.4074
F1: 0.4937
Balanced_Acc: 0.6785
Brier: 0.1038

DP difference: 0.1134
EO difference: 0.1262

Per-group metrics (Real):
      TPR (Sensitivity)  TNR (Specificity)       FPR  Precision    Recall        F1  Positive_Rate
RACE                                                                                              
0.0            0.341837           0.968504  0.031496   0.614679  0.341837  0.439344       0.071265
1.0            0.468085           0.910024  0.089976   0.634615  0.468085  0.538776       0.184615

FTU flip rate: 0.0314

Intersectional (RACE x SEX):
                          FPR       TPR        F1   Count
sensitive_feature_0                                      
0.0_1.0              0.022463  0.273381  0.368932  1430.0
0.0_2.0              0.039971  0.379447  0.475248  1629.0
1.0_1.0              0.063893  0.441989  0.526316   854.0
1.0_2.0              0.119529  0.487603  0.547564   836.0


### Train synthesizers and evaluate

In [9]:
from sdv.metadata import Metadata
from sdv.single_table import CTGANSynthesizer, GaussianCopulaSynthesizer, TVAESynthesizer

metadata = Metadata.detect_from_dataframe(df)

df_train = df.sample(frac=2/3, random_state=SEED)

sdmetrics_meta = metadata.to_dict()["tables"]["table"]

synthesizers = {
    "CTGAN": CTGANSynthesizer(
        metadata=metadata, enforce_rounding=True,
        epochs=300, verbose=True, enable_gpu=USE_GPU,
    ),
    "GaussianCopula": GaussianCopulaSynthesizer(metadata=metadata),
    "TVAE": TVAESynthesizer(
        metadata=metadata, enforce_rounding=True,
        epochs=300, enable_gpu=USE_GPU,
    ),
}

In [10]:
all_results = {}

for name, synth in synthesizers.items():
    print(f"Training {name}...")

    synth.fit(df_train)
    synth_df = synth.sample(num_rows=len(df))
    print(f"Generated {len(synth_df)} rows")

    # utility
    X_synth = synth_df.drop(columns=[TARGET])
    y_synth = (synth_df[TARGET] == "1.0").astype(int)

    model_synth = make_model(X_synth)
    model_synth.fit(X_synth, y_synth)
    y_pred_synth = model_synth.predict(X_test)
    y_proba_synth = model_synth.predict_proba(X_test)[:, 1]

    utility = compute_utility(model_synth, X_test, y_test)
    print(f"\nUtility metrics ({name}):")
    for k, v in utility.items():
        print(f"{k}: {v:.4f}")

    recall_ratio = utility["Recall"] / max(utility_real["Recall"], 1e-9)
    if recall_ratio < 0.5:
        print(f"Recall collapsed to {recall_ratio:.0%} of real baseline.")

    # fairness
    fairness = compute_fairness(
        y_test, y_pred_synth, y_proba_synth,
        sensitive_features=X_test[PROTECTED],
    )
    print(f"\nFairness metrics ({name}):")
    print(f"DP difference: {fairness['DP_difference']:.4f}")
    print(f"EO difference: {fairness['EO_difference']:.4f}")
    print(f"\nPer-group metrics:")
    print(fairness["per_group"].to_string())

    ftu = ftu_flip_rate(model_synth, X_test, PROTECTED, values=("1.0", "0.0"))
    print(f"\nFTU flip rate: {ftu:.4f}")

    # intersectional
    inter = None
    if SEX_COL:
        inter = compute_intersectional(
            y_test, y_pred_synth, X_test[PROTECTED], X_test[SEX_COL]
        )
        print(f"\nIntersectional breakdown (RACE x SEX):")
        print(inter.to_string())

    # SDMetrics
    print(f"\nSDMetrics ({name}):")
    sd_results = compute_sdmetrics(
        df, synth_df, sdmetrics_meta, TARGET, real_train_df=df_train,
    )
    for k, v in sd_results.items():
        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")

    all_results[name] = {
        "utility": utility,
        "fairness_dp": fairness["DP_difference"],
        "fairness_eo": fairness["EO_difference"],
        "ftu": ftu,
        "per_group": fairness["per_group"],
        "intersectional": inter,
        "sdmetrics": sd_results,
        "recall_ratio": recall_ratio,
    }

Training CTGAN...


Gen. (-3.48) | Discrim. (-0.48): 100%|██████████| 300/300 [17:07<00:00,  3.43s/it]


Generated 15830 rows

Utility metrics (CTGAN):
AUROC: 0.8022
Precision: 0.6039
Recall: 0.3460
F1: 0.4399
Balanced_Acc: 0.6495
Brier: 0.1123

Fairness metrics (CTGAN):
DP difference: 0.0669
EO difference: 0.0366

Per-group metrics:
      TPR (Sensitivity)  TNR (Specificity)       FPR  Precision    Recall        F1  Positive_Rate
RACE                                                                                              
0.0            0.341837           0.964754  0.035246   0.587719  0.341837  0.432258       0.074534
1.0            0.349882           0.928177  0.071823   0.619247  0.349882  0.447130       0.141420

FTU flip rate: 0.0147

Intersectional breakdown (RACE x SEX):
                          FPR       TPR        F1   Count
sensitive_feature_0                                      
0.0_1.0              0.029435  0.352518  0.433628  1430.0
0.0_2.0              0.040698  0.335968  0.431472  1629.0
1.0_1.0              0.077266  0.348066  0.425676   854.0
1.0_2.0             

Estimating Disclosure Protection (Score=0.397): 100%|██████████| 10/10 [00:33<00:00,  3.32s/it]


QualityReport_overall: 0.8947
QualityReport_column_shapes: 0.9168
QualityReport_column_pair_trends: 0.8726
DiagnosticReport_overall: 1.0000
KSComplement: 0.8126
TVComplement: 0.9278
CSTest: 0.9527
CorrelationSimilarity: 0.9588
ContinuousKLDivergence: 0.3497
ContingencySimilarity: 0.8524
DiscreteKLDivergence: 0.9302
CategoryCoverage: 1.0000
RangeCoverage: 0.9462
BoundaryAdherence: 1.0000
StatisticSimilarity: 0.9716
MissingValueSimilarity: 1.0000
GMLogLikelihood: -31.2644
LRDetection: 0.3176
SVCDetection: 0.1180
LRClassifierEfficacy: 0.5046
DTClassifierEfficacy: 0.4151
AdaBoostEfficacy: 0.4722
MLPEfficacy: 0.4317
NewRowSynthesis: 1.0000
DisclosureProtectionEstimate: 0.3972
DCRBaselineProtection: 0.2218
DCROverfittingProtection: 0.9490
CategoricalCAP: 0.3187
CategoricalZeroCAP: 0.3187
CategoricalGeneralizedCAP: 0.3187
Training GaussianCopula...
Generated 15830 rows

Utility metrics (GaussianCopula):
AUROC: 0.7964
Precision: 1.0000
Recall: 0.0025
F1: 0.0049
Balanced_Acc: 0.5012
Brier: 0.11

Estimating Disclosure Protection (Score=0.424): 100%|██████████| 10/10 [00:34<00:00,  3.42s/it]


QualityReport_overall: 0.8912
QualityReport_column_shapes: 0.9608
QualityReport_column_pair_trends: 0.8216
DiagnosticReport_overall: 1.0000
KSComplement: 0.6417
TVComplement: 0.9944
CSTest: 0.9991
CorrelationSimilarity: 0.9360
ContinuousKLDivergence: 0.1223
ContingencySimilarity: 0.9571
DiscreteKLDivergence: 0.9931
CategoryCoverage: 1.0000
RangeCoverage: 0.9725
BoundaryAdherence: 1.0000
StatisticSimilarity: 0.9333
MissingValueSimilarity: 1.0000
GMLogLikelihood: -33.0189
LRDetection: 0.4843
SVCDetection: 0.0114
LRClassifierEfficacy: 0.4887
DTClassifierEfficacy: 0.3119
AdaBoostEfficacy: 0.1422
MLPEfficacy: 0.2234
NewRowSynthesis: 1.0000
DisclosureProtectionEstimate: 0.4242
DCRBaselineProtection: 0.4073
DCROverfittingProtection: 0.9270
CategoricalCAP: 0.2842
CategoricalZeroCAP: 0.2842
CategoricalGeneralizedCAP: 0.2842
Training TVAE...
Generated 15830 rows

Utility metrics (TVAE):
AUROC: 0.8254
Precision: 0.5774
Recall: 0.4761
F1: 0.5219
Balanced_Acc: 0.7019
Brier: 0.1124

Fairness metrics

Estimating Disclosure Protection (Score=0.316): 100%|██████████| 10/10 [00:07<00:00,  1.31it/s]


QualityReport_overall: 0.9195
QualityReport_column_shapes: 0.9401
QualityReport_column_pair_trends: 0.8988
DiagnosticReport_overall: 1.0000
KSComplement: 0.8055
TVComplement: 0.9542
CSTest: 0.9768
CorrelationSimilarity: 0.9731
ContinuousKLDivergence: 0.7267
ContingencySimilarity: 0.9200
DiscreteKLDivergence: 0.9658
CategoryCoverage: 0.9724
RangeCoverage: 0.9107
BoundaryAdherence: 1.0000
StatisticSimilarity: 0.9879
MissingValueSimilarity: 1.0000
GMLogLikelihood: -29.1467
LRDetection: 0.3312
SVCDetection: 0.1757
LRClassifierEfficacy: 0.5471
DTClassifierEfficacy: 0.4453
AdaBoostEfficacy: 0.5052
MLPEfficacy: 0.5443
NewRowSynthesis: 0.9990
DisclosureProtectionEstimate: 0.3159
DCRBaselineProtection: 0.1028
DCROverfittingProtection: 0.9200
CategoricalCAP: 0.3349
CategoricalZeroCAP: 0.3349
CategoricalGeneralizedCAP: 0.3349


### Comparison table

In [11]:
summary_rows = []
for name, res in all_results.items():
    row = {"Synthesizer": name}
    row.update(res["utility"])
    row["DP_diff"] = res["fairness_dp"]
    row["EO_diff"] = res["fairness_eo"]
    row["FTU"] = res["ftu"]
    row["Quality"] = res["sdmetrics"].get("QualityReport_overall", "N/A")
    row["Disclosure"] = res["sdmetrics"].get("DisclosureProtectionEstimate", "N/A")
    row["Recall_vs_Real"] = f"{res.get('recall_ratio', 0):.0%}"
    summary_rows.append(row)

# real baseline
real_row = {"Synthesizer": "Real (baseline)"}
real_row.update(utility_real)
real_row["DP_diff"] = fairness_real["DP_difference"]
real_row["EO_diff"] = fairness_real["EO_difference"]
real_row["FTU"] = ftu_real
real_row["Quality"] = "\u2014"
real_row["Disclosure"] = "\u2014"
real_row["Recall_vs_Real"] = "100%"
summary_rows.insert(0, real_row)

summary_df = pd.DataFrame(summary_rows).set_index("Synthesizer")
float_cols = summary_df.select_dtypes(include="number").columns
summary_df[float_cols] = summary_df[float_cols].round(4)

print(summary_df.to_string())
summary_df.to_csv("meps_summary.csv")

                  AUROC  Precision  Recall      F1  Balanced_Acc   Brier  DP_diff  EO_diff     FTU   Quality Disclosure Recall_vs_Real
Synthesizer                                                                                                                           
Real (baseline)  0.8426     0.6264  0.4074  0.4937        0.6785  0.1038   0.1134   0.1262  0.0314         —          —           100%
CTGAN            0.8022     0.6039  0.3460  0.4399        0.6495  0.1123   0.0669   0.0366  0.0147  0.894699   0.397198            85%
GaussianCopula   0.7964     1.0000  0.0025  0.0049        0.5012  0.1193   0.0012   0.0047  0.0008  0.891202   0.424174             1%
TVAE             0.8254     0.5774  0.4761  0.5219        0.7019  0.1124   0.1882   0.2636  0.0651   0.91945   0.315855           117%


### Per-group FPR comparison

In [13]:
fpr_comparison = {}
fpr_comparison["Real"] = fairness_real["per_group"]["FPR"]
for name, res in all_results.items():
    fpr_comparison[name] = res["per_group"]["FPR"]

fpr_df = pd.DataFrame(fpr_comparison)
print("\nFalse Positive Rate by RACE group:")
print(fpr_df.to_string())

# intersectional FPR comparison
if SEX_COL:
    print("\n\nIntersectional FPR (RACE x SEX):")
    inter_fpr = {"Real": inter_real["FPR"]}
    for name, res in all_results.items():
        if res["intersectional"] is not None:
            inter_fpr[name] = res["intersectional"]["FPR"]
    inter_fpr_df = pd.DataFrame(inter_fpr)
    print(inter_fpr_df.to_string())
    inter_fpr_df.to_csv("meps_intersectional_fpr.csv")



False Positive Rate by RACE group:
          Real     CTGAN  GaussianCopula      TVAE
RACE                                              
0.0   0.031496  0.035246             0.0  0.035621
1.0   0.089976  0.071823             0.0  0.149171


Intersectional FPR (RACE x SEX):
                         Real     CTGAN  GaussianCopula      TVAE
sensitive_feature_0                                              
0.0_1.0              0.022463  0.029435             0.0  0.024787
0.0_2.0              0.039971  0.040698             0.0  0.045785
1.0_1.0              0.063893  0.077266             0.0  0.132244
1.0_2.0              0.119529  0.065657             0.0  0.168350


# Disabled: Toy Data + Adult Dataset

# Disabled: Intersectional Bias Assessment (IBA) Dataset